# 约束满足问题（CSP）：来排一排吃完便当的顺序

## 这一章要弄清楚的三件事

**约束满足问题**（Constraint Satisfaction Problem，简称 **CSP**）是这一章的主角。

- 它到底是一类什么样的问题
- 穷举（把所有情况都列出来）为什么很快就会算不动
- 换成 Z3 以后，凭什么就轻松了

一路下来只会碰到两个词：变量、约束。别的先不用管，放松往下读。

## 到底什么叫约束满足问题

说白了，就是**有几个变量，要给它们各定一个值，而且这些值必须满足给定的条件（约束）**。

空讲不好懂，先看一个小小的例子。

> 5 个小朋友（A, B, C, D, E）一个接一个吃完了便当。
> 根据下面这几句话，把他们吃完的先后顺序排出来。
>
> - A 比 B 吃得晚，可比 C 吃得早。
> - D 比 E 吃得早，可比 C 吃得晚。

对着这个例子，两样东西就浮出来了。

- **变量**：5 个人各自“第几个吃完”
- **约束**：“A 在 B 之后”“A 在 C 之前”这类条件

你把 5 个人的顺序排定，回头逐条检查、一条也没破，这个排列就是答案。

## 先把这些条件写成符号

条件用大白话摆在那里，不好想，我们换成符号再写一遍。

记 $X \prec Y$ 为“X 比 Y 先吃完”。

1. A 比 B 晚、比 C 早 → $B \prec A \prec C$
2. D 比 E 早、比 C 晚 → $C \prec D \prec E$

两条一接，就串成了 $B \prec A \prec C \prec D \prec E$ 这一条链。

于是要找的东西变得很具体：满足这个先后顺序的那个排列。

## 那就先上笨办法：全部试一遍

最不需要动脑子的做法，是**把可能的顺序全都摆出来，再一条条筛掉不合格的**。

5 个人排成一列，总共只有 $5! = 120$ 种。就算一种一种试，也是一眨眼的事。

In [1]:
from itertools import permutations

kids = ["A", "B", "C", "D", "E"]

# order 是“按吃完顺序排列的小孩编号”
for order in permutations(range(len(kids))):
    rank = {kids[i]: pos for pos, i in enumerate(order)}   # 每个小孩的名次
    if rank["B"] < rank["A"] < rank["C"] and rank["C"] < rank["D"] < rank["E"]:
        print(" → ".join(kids[i] for i in order))

B → A → C → D → E


筛完只剩**B → A → C → D → E** 这一个顺序。

5 个人，眨眼就出来了。可要是人数往上翻呢？排列的数目是按 $n!$ 涨的，一涨就收不住。

In [2]:
import math

for n in [5, 8, 10, 12]:
    print(f"{n:2d} 人: {math.factorial(n):,} 种")

 5 人: 120 种
 8 人: 40,320 种
10 人: 3,628,800 种
12 人: 479,001,600 种


12 个人就是大约 4 亿 8 千万种。哪怕一秒能试 100 万种，也得花上 8 分钟。再往上到 20 个人（约 2.4×10¹⁸ 种），这辈子都等不到结果。

“全部试一遍”这条路，人数稍微加几个就走不通了。这一章后半段，我们看 Z3 怎么绕过这堵墙。

### 顺带说说 `itertools.permutations`

穷举时用的 `permutations()`，干的事是把给它的元素**按各种顺序全排一遍**。

```python
itertools.permutations(iterable, r=None)
```

- `iterable`：要排的那些元素
- `r`：一次取几个（不写就是全取）

给三个元素，出来的就是 $3! = 6$ 种。

In [3]:
for p in permutations(["A", "B", "C"]):
    print(p)

('A', 'B', 'C')
('A', 'C', 'B')
('B', 'A', 'C')
('B', 'C', 'A')
('C', 'A', 'B')
('C', 'B', 'A')


用 `permutations()` 之前，有三件事得心里有数。

1. **顺序算不同**：(A, B) 和 (B, A) 是两个结果。
2. **它不管重复**：元素里本来就有相同的，同一个排列会冒出来好几遍。
3. **复杂度是 $n!$**：元素一多，数目就炸了。

拿它盯一盯小问题、用眼睛确认答案，挺好用；真碰上大问题就别指望它。这时候，该把活交给求解器。

## 换个工具：Z3

Z3 属于**约束求解器**这一类。你把条件原样写进去，它自己去找哪一组取值能同时满足。

跟穷举比，差别在这里。

| | 各自要干的事 |
|---|---|
| 穷举 | 把所有候补摆出来，一个一个验 |
| Z3 | 只把条件写清楚，找的事交给它 |

不管解什么题，用 Z3 的步骤都是这三步。

1. 建变量
2. 加约束
3. 取解

In [4]:
from z3 import IntVector, Solver, Distinct, sat

kids = ["A", "B", "C", "D", "E"]

# 表示每个小孩“吃完的名次”的 5 个整数变量（pos[0] 是 A，pos[1] 是 B …）
pos = IntVector("pos", 5)
eat_a, eat_b, eat_c, eat_d, eat_e = pos

In [5]:
solver = Solver()

# 1. 名次是 0 到 4 中的一个
for p in pos:
    solver.add(p >= 0, p < len(kids))

# 2. 没有名次相同的小孩
solver.add(Distinct(pos))

# 3. 问题的条件  B ≺ A ≺ C ≺ D ≺ E
solver.add(eat_b < eat_a, eat_a < eat_c)
solver.add(eat_c < eat_d, eat_d < eat_e)

# 4. 取出解，按名次从小到大排列
if solver.check() == sat:
    model = solver.model()
    rank = {kid: model[pos[i]].as_long() for i, kid in enumerate(kids)}
    print(" → ".join(sorted(kids, key=lambda kid: rank[kid])))

B → A → C → D → E


跑出来的答案跟穷举一模一样。这里值得停一下，看看可搜索的范围是怎么一层层收窄的。

1. 只建 `IntVector("pos", 5)`，值随便取 → 无限
2. 补上 `0 <= p < 5` → $5^5 = 3125$ 种
3. 再补 `Distinct(pos)` → $5! = 120$ 种（跟穷举站在同一条线上）
4. 最后补上 $B \prec A \prec C \prec D \prec E$ → 1 种

穷举是从第 3 步那个范围起步的，剩下的路自己一条条走完。Z3 反过来：条件先到手，那些注定不合格的候补，它压根不去造。人再多，套路还是这一套。

## 收个尾

| 手法 | 计算量 | 适合什么情形 |
|---|---|---|
| `itertools.permutations` | $O(n!)$ | 小问题，用眼睛核对 |
| Z3 | 从条件出发高效搜索 | 条件绕、人数多的题 |

- 约束满足问题，说到底就是“变量”加“约束”两样东西
- 穷举要 $n!$ 这么多步，规模稍大就废了
- Z3 能照原样收下约束，题换了也改得动